# 📝 LangChain 에이전트와 도구 과제 LV1 정답 — 도구 정의·명세·에이전트 궤적 (강사용)

각 문제의 **모범답안 + 해설**입니다.

- 도구를 **작성**하는 문제는 도구를 **직접 호출**해 채점합니다(모델을 거치지 않아 결정적).
- **모델을 부르는 문제**(에이전트 궤적·구조화 출력)는 **타입·구조**(도구가 최소 1회 불렸는지, 답이 스키마 객체로 왔는지)로만 채점합니다 — 도구 호출 횟수와 답 문장은 실행마다 달라집니다.

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 — 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day19_LangChain_에이전트_툴/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다 — 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 — 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 이 과제에서 쓸 도구 두 가지 — 에이전트 문제에 이 도구들을 씁니다.
from langchain_core.tools import tool


@tool
def shipping_fee(order_amount: int) -> int:
    """주문 금액(order_amount)을 받아 배송비를 돌려준다. 3만원 이상이면 무료(0), 그 미만이면 3000원."""
    return 0 if order_amount >= 30000 else 3000


@tool
def order_total(unit_price: int, quantity: int) -> int:
    """단가(unit_price)와 수량(quantity)을 받아 주문 총액을 돌려준다."""
    return unit_price * quantity


print("도구 준비:", shipping_fee.name, "/", order_total.name)

In [ ]:
# [제공 코드] 역할(페르소나)을 정하는 시스템 프롬프트 — 6번에서 씁니다.
PERSONA_POLITE = "너는 친절한 온라인 서점 상담원이다. 항상 정중한 존댓말로 답한다."

In [ ]:
# [제공 코드] 에이전트 문제에서 공통으로 쓰는 도구들
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 1. 도구 만들기 — 할인가 계산
**배경**: 온라인 서점의 할인가를 계산하는 도구를 만듭니다.

**요구사항**:
- `@tool` 을 붙인 함수 **`discount_price(price: int, rate: int) -> int`** 를 정의하세요.
- 정가 `price` 에서 `rate` 퍼센트를 할인한 **정수** 가격을 돌려줍니다(예: 20000원에서 10% → 18000).
- docstring 에 도구가 하는 일을 **한국어로** 적으세요. 이 docstring 이 곧 **모델이 읽는 도구 설명**이므로 무엇을 받아 무엇을 돌려주는지 알 수 있게 **열 글자 이상**의 한 문장으로 적습니다(채점이 길이를 확인합니다).

**예시**: `discount_price.invoke({'price': 20000, 'rate': 10})` → `18000`

<details><summary>힌트</summary>

```text
접근방법:
- 할인가 = price 에서 (price 의 rate 퍼센트)를 뺀 값. 정수로 맞춘다.

세부구현:
1. 함수 위에 @tool 을 붙이고 매개변수·반환에 타입힌트를 단다.
2. docstring 으로 도구 설명을 적는다.
3. 할인액을 구해 정가에서 뺀다. 결과가 정수가 되도록 나눗셈에 주의한다.
```

</details>

In [ ]:
@tool
def discount_price(price: int, rate: int) -> int:
    # docstring 은 주석이 아니라 '모델이 읽는 도구 설명'이다 — 빠지면 @tool 이 에러를 낸다
    """정가(price)에서 rate 퍼센트를 할인한 정수 가격을 돌려준다."""
    # // 로 나눠야 정수가 나온다(/ 를 쓰면 18000.0 이라 채점의 == 18000 이 어긋난다)
    return price - price * rate // 100


# 도구는 결국 함수 — 모델 없이 직접 불러 확인할 수 있다
print(discount_price.invoke({'price': 20000, 'rate': 10}))

In [ ]:
# [자가채점]
assert discount_price.name == 'discount_price'
# docstring 이 곧 모델에게 가는 설명이다 — 한 글자짜리 자리채움은 도구를 못 고르게 만든다
assert len(discount_price.description.strip()) >= 10, \
    'docstring 에 하는 일을 한 문장(열 글자 이상)으로 적으세요 — 모델이 읽는 설명입니다'
assert discount_price.invoke({'price': 20000, 'rate': 10}) == 18000
assert discount_price.invoke({'price': 15000, 'rate': 20}) == 12000
print('✅ 통과!')

**해설**: `@tool` 은 함수를 도구로 바꿉니다. 도구는 결국 함수라 `.invoke({인자})` 로 직접 실행해 확인할 수 있습니다. **흔한 실수**: 반환을 실수(float)로 두면 채점이 어긋납니다 — `//` 로 정수 나눗셈을 쓰세요.

## 2. 도구 만들기 — 제목 정리
**배경**: 사용자가 입력한 책 제목에는 앞뒤 공백이나 중간에 여러 칸의 공백이 섞이곤 합니다. 이를 깔끔히 다듬는 도구를 만듭니다(1번과 다른 유형 — 문자열 처리).

**요구사항**:
- `@tool` 을 붙인 함수 **`clean_title(text: str) -> str`** 를 정의하세요.
- 앞뒤 공백을 없애고, 문자열 **중간의 연속된 공백을 한 칸**으로 줄여 돌려줍니다.
- 1번과 마찬가지로 docstring 에 하는 일을 **한국어 한 문장(열 글자 이상)** 으로 적으세요.

**예시**: `clean_title.invoke({'text': '  파이썬   입문  '})` → `'파이썬 입문'`

<details><summary>힌트</summary>

```text
접근방법:
- 문자열을 공백 기준으로 쪼갠 뒤 한 칸으로 다시 잇는다.

세부구현:
1. @tool 과 타입힌트·docstring 을 단다.
2. text.split() 은 연속 공백을 무시하고 단어 리스트를 준다.
3. 쪼갠 단어들을 한 칸 공백으로 다시 이어 붙여 반환한다.
```

</details>

In [ ]:
@tool
def clean_title(text: str) -> str:
    """책 제목 문자열의 앞뒤 공백을 없애고 중간 연속 공백을 한 칸으로 줄여 돌려준다."""
    # split() 에 인자를 주지 않으면 연속 공백을 하나로 보고 쪼갠다 — strip 과 중간 공백 정리가 한 번에 된다
    # (text.split(' ') 로 쓰면 빈 문자열이 섞여 정리가 안 된다)
    return ' '.join(text.split())


print(clean_title.invoke({'text': '  파이썬   입문  '}))

In [ ]:
# [자가채점]
assert clean_title.name == 'clean_title'
assert len(clean_title.description.strip()) >= 10, \
    'docstring 에 하는 일을 한 문장(열 글자 이상)으로 적으세요'
assert clean_title.invoke({'text': '  파이썬   입문  '}) == '파이썬 입문'
assert clean_title.invoke({'text': '데이터  분석 기초'}) == '데이터 분석 기초'
print('✅ 통과!')

**해설**: `text.split()` 은 인자를 주지 않으면 **연속 공백을 하나로** 보고 쪼개므로, `' '.join(...)` 과 함께 쓰면 공백 정리가 한 줄로 됩니다. 도구 입력을 이렇게 다듬어 두면 뒤 단계가 안정됩니다.

## 3. 도구 명세 읽기
**배경**: 모델은 도구의 **이름·설명·인자 명세**를 보고 도구를 고릅니다. 제공된 `order_total` 도구의 명세를 직접 읽어 봅니다.

**요구사항**:
- 제공된 `order_total` 도구에서 **이름**을 변수 **`tool_name`**, **설명**을 **`tool_desc`**, **인자 이름 목록**을 **`arg_names`** 에 담으세요.
- 이름은 `.name`, 설명은 `.description` 에 있습니다. **인자 정보를 담은 속성이 하나 더** 있으니 찾아서 그 **열쇠(키) 목록**을 리스트로 만드세요.

**예시**: `tool_name` 은 `'order_total'`, `arg_names` 는 `['unit_price', 'quantity']` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 세 값 모두 도구 객체가 이미 갖고 있다 — 손으로 적지 말고 속성에서 읽어 온다.

세부구현:
1. 이름과 설명은 각각 name·description 속성에서 읽는다.
2. 인자 명세는 '인자 이름 -> 명세' 형태의 딕셔너리다. 그 딕셔너리의 열쇠만 모아
   리스트로 만들면 인자 이름 목록이 된다.
```

</details>

In [ ]:
# 세 값 모두 도구 객체에서 '읽어' 온다 — 손으로 적으면 도구를 고치는 순간 어긋난다
tool_name = order_total.name
tool_desc = order_total.description        # docstring 이 그대로 설명이 된다
arg_names = list(order_total.args.keys())  # args 는 '인자 이름 -> 명세' 딕셔너리라 keys 만 뽑는다
print(tool_name, '/', arg_names)
print(tool_desc)

In [ ]:
# [자가채점]
# 값을 손으로 적지 않고 도구에서 읽어 왔는지 — 도구 객체와 그대로 일치해야 한다
assert tool_name == order_total.name
assert tool_desc == order_total.description
assert arg_names == list(order_total.args.keys())
assert tool_name == 'order_total'
assert arg_names == ['unit_price', 'quantity']
print('✅ 통과!')

**해설**: 이 세 가지(`name`·`description`·`args`)가 **모델에게 전달되는 도구 명세**입니다. 설명이 비어 있거나 모호하면 모델이 도구를 잘못 고릅니다.

**채점 기준**: 값을 문자열로 **손수 적어도 눈으로는 같아 보이므로**, 채점은 세 변수가 `order_total` 객체의 값과 **그대로 일치하는지**를 봅니다. 도구의 docstring 을 한 글자 고치면 손으로 적은 답은 바로 어긋납니다 — 명세는 **읽어 오는 것**이지 옮겨 적는 것이 아닙니다.

## 4. 에이전트 궤적 — 배송비 도구
**배경**: `shipping_fee` 도구 하나를 가진 에이전트에게 배송비를 묻고, **어떤 도구가 불렸는지**와 **도구 결과**를 궤적에서 읽습니다.

**요구사항**:
- `create_agent(model, [shipping_fee])` 로 에이전트를 만들고, 질문 **"2만원어치 책을 주문하면 배송비가 얼마인가요?"** 로 `invoke` 한 결과를 변수 **`res4`** 에 담으세요.
- `res4['messages']` 에서 **ToolMessage** 만 골라 변수 **`tool_msgs4`** 에 담으세요.

**예시**: `tool_msgs4` 에는 `shipping_fee` 의 실행 결과가 담깁니다. 2만원은 3만원 미만이라 보통 `'3000'` 이 나옵니다(도구를 몇 번 부를지는 모델이 정하므로 채점은 **최소 1회 불렸는지**만 봅니다).

<details><summary>힌트</summary>

```text
접근방법:
- 에이전트를 만들고 invoke 한 뒤, 메시지에서 ToolMessage 만 걸러 낸다.

세부구현:
1. create_agent 에 model 과 [shipping_fee] 를 넘겨 에이전트를 만든다.
2. messages 에 role=user, content=질문을 넣어 invoke 하고 res4 에 담는다.
3. isinstance(m, ToolMessage) 인 메시지만 모아 tool_msgs4 에 담는다.
```

</details>

In [ ]:
agent4 = create_agent(model, [shipping_fee])
# messages 는 리스트 — 대화를 통째로 넘기는 형식이라 질문 하나여도 리스트로 감싼다
res4 = agent4.invoke({'messages': [{'role': 'user', 'content': '2만원어치 책을 주문하면 배송비가 얼마인가요?'}]})
# 궤적에서 '도구 실행 결과'만 골라 낸다(도구 호출 결정은 그 앞 AIMessage 에 있다)
tool_msgs4 = [m for m in res4['messages'] if isinstance(m, ToolMessage)]
print([m.name for m in tool_msgs4], [m.content for m in tool_msgs4])

In [ ]:
# [자가채점]
# tool_msgs4 가 정말 res4 의 궤적에서 나왔는지 먼저 확인한다(직접 만든 리스트면 여기서 걸린다)
assert tool_msgs4 == [m for m in res4['messages'] if isinstance(m, ToolMessage)]
assert res4['messages'][0].text.strip() == '2만원어치 책을 주문하면 배송비가 얼마인가요?', '지문의 질문을 그대로 넣어 실행하세요'
assert len(tool_msgs4) >= 1, '도구가 한 번도 불리지 않았습니다'
assert all(m.name == 'shipping_fee' for m in tool_msgs4)
print('✅ 통과!')

**해설**: 에이전트가 `shipping_fee` 를 부르고(2만원 → 3000원), 그 결과가 **ToolMessage** 로 남습니다. 우리는 루프를 돌리지 않았지만 에이전트가 알아서 도구를 실행했습니다.

**채점 기준**: 실제 모델을 부르므로 **도구를 몇 번 부를지는 모델이 정합니다** — 확인 삼아 두 번 부를 수도 있습니다. 그래서 `== ['shipping_fee']`(정확히 한 번)가 아니라 **최소 1회 불렸는지**와 **불린 도구가 그 도구인지**만 채점합니다. 도구가 돌려준 값도 모델이 넘긴 인자에 달려 있어 고정하지 않습니다.

## 5. 도구가 필요 없는 질문
**배경**: 모든 질문에 도구가 필요한 것은 아닙니다. 일반적인 질문에는 에이전트가 도구를 **부르지 않습니다**.

**요구사항**:
- `create_agent(model, [shipping_fee])` 로 에이전트를 만들고, 질문 **"온라인 서점은 어떤 곳인가요? 한 문장으로 알려줘."** 로 `invoke` 한 결과를 변수 **`res5`** 에 담으세요.
- 궤적의 **ToolMessage 개수**를 변수 **`n_tools5`** 에 담으세요.

**예시**: 이 질문에는 도구가 필요 없어 `n_tools5` 은 **0** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- invoke 한 뒤 ToolMessage 개수를 센다.

세부구현:
1. 에이전트를 만들고 질문을 그대로 invoke 해 res5 에 담는다.
2. isinstance(m, ToolMessage) 인 메시지 수를 세어 n_tools5 에 담는다.
```

</details>

In [ ]:
agent5 = create_agent(model, [shipping_fee])
res5 = agent5.invoke({'messages': [{'role': 'user', 'content': '온라인 서점은 어떤 곳인가요? 한 문장으로 알려줘.'}]})
# 개수를 '세는' 것이 이 문제의 핵심 — 0 을 손으로 적으면 궤적을 본 것이 아니다
n_tools5 = sum(1 for m in res5['messages'] if isinstance(m, ToolMessage))
print('도구 사용 횟수:', n_tools5)
print('최종 답:', res5['messages'][-1].text)

In [ ]:
# [자가채점]
# 0 은 손으로도 적을 수 있는 값이라, res5 를 실제로 실행했는지부터 확인한다
assert res5['messages'][0].text.strip() == '온라인 서점은 어떤 곳인가요? 한 문장으로 알려줘.', '지문의 질문을 그대로 넣어 실행하세요'
assert isinstance(res5['messages'][-1], AIMessage) and res5['messages'][-1].text.strip()
assert n_tools5 == sum(1 for m in res5['messages'] if isinstance(m, ToolMessage))
assert n_tools5 == 0, '배송비와 무관한 질문이라 도구가 불리지 않아야 합니다'
print('✅ 통과!')

**해설**: 에이전트는 도구가 **필요한지 아닌지**를 먼저 판단합니다. 배송비와 무관한 일반 질문에는 도구를 건너뛰고 바로 답하므로 ToolMessage 가 없습니다.

**채점 기준**: `n_tools5 = 0` 은 **손으로도 적을 수 있는 값**이라, 채점이 먼저 `res5` 를 들여다봅니다 — 지문의 질문이 그대로 들어갔는지, 최종 답이 실제로 왔는지, 그리고 `n_tools5` 이 **그 궤적을 세어 나온 수와 같은지**. 셋을 모두 통과해야 0 이 의미를 갖습니다.

이 과제에서 도구 **선택 결과**에 값을 거는 채점은 여기 하나뿐입니다 — '도구를 안 쓴다'가 곧 이 문제가 가르치는 내용이기 때문입니다. 붙어 있는 도구가 `shipping_fee` 하나뿐이고 질문에 금액이 전혀 없어 모델이 부를 인자조차 없습니다. 다른 문제들처럼 여러 도구를 두고 **어느 것을 골랐는지**를 채점하지는 않습니다(그건 실행마다 달라질 수 있습니다).

## 6. 역할 부여 — system_prompt 로 상담원 페르소나
**배경**: `system_prompt` 로 에이전트에 **역할**을 줄 수 있습니다. 역할을 줘도 도구 사용은 그대로 작동하는지 확인합니다.

**요구사항**:
- `create_agent(model, [shipping_fee], system_prompt=PERSONA_POLITE)` 로 에이전트를 만들고, 질문 **"주문 금액 25000원의 배송비를 계산해줘."** 로 `invoke` 한 결과를 변수 **`res6`** 에 담으세요.
- 궤적의 **ToolMessage** 만 골라 **`tool_msgs6`** 에 담으세요.

**예시**: 2만5천원은 3만원 미만이라 도구 결과는 보통 `'3000'` 입니다. 최종 답의 **말투**가 정중해졌는지 눈으로 확인하세요.

<details><summary>힌트</summary>

```text
접근방법:
- create_agent 에 system_prompt 인자를 더해 만든다.

세부구현:
1. create_agent 의 세 번째 인자로 system_prompt=PERSONA_POLITE 를 준다.
2. 질문을 그대로 invoke 해 res6 에 담는다.
3. ToolMessage 만 골라 tool_msgs6 에 담는다.
```

</details>

In [ ]:
# system_prompt 는 대화 맨 앞에 붙는 '역할 지시' — 도구 목록과는 별개의 인자다
agent6 = create_agent(model, [shipping_fee], system_prompt=PERSONA_POLITE)
res6 = agent6.invoke({'messages': [{'role': 'user', 'content': '주문 금액 25000원의 배송비를 계산해줘.'}]})
tool_msgs6 = [m for m in res6['messages'] if isinstance(m, ToolMessage)]
print('도구 결과:', [m.content for m in tool_msgs6])
# 역할이 바뀐 것은 도구가 아니라 '최종 답의 말투' — 눈으로 확인할 자리다
print('최종 답:', res6['messages'][-1].text)

In [ ]:
# [자가채점]
assert tool_msgs6 == [m for m in res6['messages'] if isinstance(m, ToolMessage)]
assert res6['messages'][0].text.strip() == '주문 금액 25000원의 배송비를 계산해줘.', '지문의 질문을 그대로 넣어 실행하세요'
assert len(tool_msgs6) >= 1, '역할을 줘도 도구는 그대로 불려야 합니다'
assert all(m.name == 'shipping_fee' for m in tool_msgs6)
assert isinstance(res6['messages'][-1].text, str)
print('✅ 통과!')

**해설**: `system_prompt` 로 역할을 줘도 **도구 판단은 그대로** 작동합니다. 역할은 주로 **말투·태도·답변 범위**를 바꿉니다. 최종 답의 말투가 정중해졌는지 확인해 보세요.

## 7. 구조화 출력 — 스키마를 씌운 부품으로 리뷰 두 건 처리
**배경**: 리뷰가 쌓이면 사람이 다 읽을 수 없어 **감성과 별점을 데이터로** 뽑아 둡니다. 원하는 출력 모양을 `pydantic` 스키마로 적는 법은 이미 익혔으니, 여기서는 그 스키마를 **모델에 씌우는** 일을 합니다. `model.with_structured_output(스키마)` 는 모델을 바꾸는 것이 아니라 **스키마를 씌운 새 부품**을 돌려주고, 그 부품을 `invoke` 하면 답이 문장이 아니라 **객체**로 옵니다.

아래 **제공 셀**의 스키마 `ReviewSummary` 와 리뷰 두 건(`REVIEW_A`·`REVIEW_B`)을 씁니다(실행만 하세요). **스키마를 먼저 읽어 보세요** — `Literal` 이 감성을 세 값으로 **좁혀** 모델이 '보통' 같은 딴 값을 낼 여지를 없애고, `Field(description=...)` 이 각 칸의 뜻을 알려 줍니다. 이 설명은 사람이 보는 주석이 아니라 **스키마와 함께 모델에게 통째로 전달되는 출력 설명서**입니다.

In [ ]:
# [제공 코드] 리뷰 요약 스키마와 리뷰 두 건 — 7번에서 씁니다. 이 셀은 실행만 하세요.
from typing import Literal

from pydantic import BaseModel, Field


class ReviewSummary(BaseModel):
    """도서 리뷰의 감성과 별점."""

    # Literal 은 값을 세 가지로 좁힌다 — 모델이 '보통' 같은 딴 값을 낼 여지를 없앤다
    sentiment: Literal["긍정", "부정", "중립"] = Field(description="리뷰의 전체 감성")
    # description 은 사람용 주석이 아니라 스키마와 함께 모델에게 전달되는 설명이다
    stars: int = Field(description="1~5 사이의 별점")


REVIEW_A = ("배송이 하루 만에 왔고 설명이 아주 쉬워서 술술 읽혔습니다. "
            "예제도 실무에 바로 쓸 만해서 주변에 계속 추천하고 있어요.")
REVIEW_B = ("오탈자가 페이지마다 보이고 예제 코드는 그대로 따라 해도 돌아가지 않습니다. "
            "배송도 일주일이나 걸려서 여러모로 실망했습니다.")

print("스키마와 리뷰 준비 완료:", list(ReviewSummary.model_fields))

**요구사항**:

- `model.with_structured_output(ReviewSummary)` 가 돌려주는 부품을 변수 **`reviewer`** 에 담으세요.
- 그 **부품 하나를 두 리뷰에 다시 써서** `REVIEW_A` 의 결과를 **`summary_a`**, `REVIEW_B` 의 결과를 **`summary_b`** 에 담으세요. 리뷰마다 부품을 새로 만들지 않습니다 — **한 번 만들어 여러 입력에 재사용**하는 것이 이 문제의 핵심입니다.
- 두 결과의 `sentiment` 와 `stars` 를 출력해 눈으로 확인하세요.

**예시**: `summary_a` 는 문자열이 아니라 `ReviewSummary` 객체라 `summary_a.sentiment`·`summary_a.stars` 처럼 **점(.)으로 칸을 바로 꺼내** 씁니다. `REVIEW_A` 는 칭찬 일색, `REVIEW_B` 는 불만 일색이라 **`summary_a.stars` 가 `summary_b.stars` 보다 높게** 나옵니다.

> 실제 모델이 판단하므로 채점은 **타입·구조와 방향**으로만 합니다 — 감성 값과 별점 숫자는 실행할 때마다 조금씩 달라질 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델에 스키마를 씌워 부품을 하나 만들어 변수에 담고, 그 변수를 두 번 invoke 한다.

세부구현:
1. 모델에 ReviewSummary 를 씌운 결과를 reviewer 에 담는다.
2. reviewer 에 첫 리뷰를 넣어 나온 결과를 summary_a 에 담는다.
3. 같은 reviewer 에 둘째 리뷰를 넣어 나온 결과를 summary_b 에 담는다.
4. 두 결과에서 감성과 별점 칸을 꺼내 출력한다.
```

</details>

In [ ]:
# 모델이 바뀌는 것이 아니라 '스키마를 씌운 새 부품'이 나온다 — model 은 그대로 남아 다른 문제에서 계속 쓴다
reviewer = model.with_structured_output(ReviewSummary)

# 부품 하나를 두 입력에 다시 쓴다 — 리뷰가 100건이어도 만드는 것은 한 번뿐이다
summary_a = reviewer.invoke(REVIEW_A)
summary_b = reviewer.invoke(REVIEW_B)

# 돌아온 것은 문장이 아니라 객체다 — 문장에서 숫자를 뽑아내는 파싱이 필요 없다
print('A:', summary_a.sentiment, summary_a.stars)
print('B:', summary_b.sentiment, summary_b.stars)

In [ ]:
# [자가채점]
# reviewer 는 model 에 스키마를 씌운 '부품' 이다 — model 을 그대로 담아 두면 답이 객체로 오지 않는다
assert reviewer is not model and hasattr(reviewer, 'invoke'), \
    'reviewer 에는 with_structured_output 이 돌려준 부품을 담으세요'
# 이 문제의 핵심 — 답이 문장이 아니라 스키마 객체로 왔는가
assert isinstance(summary_a, ReviewSummary), 'summary_a 가 ReviewSummary 객체가 아닙니다'
assert isinstance(summary_b, ReviewSummary), 'summary_b 가 ReviewSummary 객체가 아닙니다'
assert summary_a is not summary_b, '두 리뷰를 각각 invoke 해 따로 받으세요'
# Literal 이 값을 세 가지로 좁혀 두었다 — 스키마가 보장하는 범위
assert summary_a.sentiment in {'긍정', '부정', '중립'}
assert summary_b.sentiment in {'긍정', '부정', '중립'}
assert 1 <= summary_a.stars <= 5 and 1 <= summary_b.stars <= 5
# 숫자는 모델이 정하므로 값을 못박지 않고 방향만 본다
assert summary_a.stars > summary_b.stars, '칭찬 리뷰의 별점이 불만 리뷰보다 높아야 합니다'
print('✅ 통과!')

**해설**: `with_structured_output(ReviewSummary)` 는 **모델을 바꾸지 않습니다** — 스키마를 씌운 **새 부품**을 돌려줄 뿐이라 `model` 은 그대로 남아 다른 문제에서 계속 쓸 수 있습니다. 그 부품을 부르면 답이 `ReviewSummary` **객체**로 와서 `summary_a.stars` 처럼 바로 꺼내 씁니다 — 문장에서 숫자를 찾아내는 파싱이 통째로 사라집니다.

**부품은 한 번만 만듭니다**: 리뷰가 100건이어도 `reviewer` 하나를 100번 부르면 됩니다. **흔한 실수**: 리뷰마다 `model.with_structured_output(...)` 를 다시 부르는 것 — 결과는 같지만 같은 부품을 계속 새로 조립하는 셈입니다.

**스키마가 왜 이렇게 생겼나**: 제공 셀의 `ReviewSummary` 를 다시 읽어 보세요. `Literal` 로 감성을 세 값으로 **좁히면** 모델이 '보통'·'그럭저럭' 같은 딴 값을 낼 여지가 없어집니다. `Field(description=...)` 은 사람용 주석이 아니라 **스키마와 함께 모델에게 전달되는 설명**이라, 비워 두면 칸만 있고 뜻이 없어 모델이 무엇을 채울지 모릅니다. 선언한 스키마는 `ReviewSummary.model_fields` 로 필드를, `typing.get_args(...)` 로 `Literal` 의 허용값을 들여다볼 수 있습니다 — 모델에게 무엇이 전달되는지 눈으로 확인하는 방법입니다.

**채점 기준**: 감성과 별점은 모델이 정하므로 값을 못박지 않습니다. 대신 **답이 객체로 왔는지**(타입)와 **칭찬 리뷰의 별점이 불만 리뷰보다 높은지**(방향)를 봅니다. 스키마가 보장하는 것은 **내용이 아니라 모양**이라는 점이 여기서 그대로 드러납니다.

## 8. 메시지 궤적의 구조 이해
**배경**: 에이전트의 궤적은 **사람 질문 → AI 도구호출 → 도구결과 → AI 최종답** 순서로 쌓입니다. 4번의 궤적(`res4`)을 이용해 각 메시지 종류의 개수를 세어 봅니다.

**요구사항**:
- `res4['messages']` 에서 **HumanMessage 개수**를 **`n_human`**, **ToolMessage 개수**를 **`n_tool`**, **AIMessage 개수**를 **`n_ai`** 에 담으세요.
- 아래 서술 셀에 **각 메시지 종류가 궤적에서 어떤 역할**을 하는지 한두 문장으로 적으세요.

**예시**: 보통 이 궤적에는 사람 질문 1개, 도구 결과 1개, AI 메시지 2개(도구를 부르는 AI + 최종 답 AI)가 있습니다. 모델이 도구를 한 번 더 부르면 도구 결과와 AI 메시지가 함께 늘어납니다 — 그래서 채점은 **사람 질문 1개 / 도구 결과 1개 이상 / AI 메시지 2개 이상**으로 봅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 메시지 리스트를 돌며 isinstance 로 종류별 개수를 센다.

세부구현:
1. 메시지를 돌며 HumanMessage·ToolMessage·AIMessage 각각의 개수를 센다.
2. 세 변수에 담아 출력한다.
```

</details>

In [ ]:
# 4번에서 만든 궤적을 다시 쓴다 — 새로 부르면 호출 비용만 늘고 세는 대상이 달라진다
n_human = sum(1 for m in res4['messages'] if isinstance(m, HumanMessage))
n_tool = sum(1 for m in res4['messages'] if isinstance(m, ToolMessage))
n_ai = sum(1 for m in res4['messages'] if isinstance(m, AIMessage))
print('사람:', n_human, '/ 도구:', n_tool, '/ AI:', n_ai)

In [ ]:
# [자가채점]
# 세 수가 손으로 적은 값이 아니라 res4 를 실제로 센 값인지 먼저 확인한다
msgs4 = res4['messages']
assert n_human == sum(1 for m in msgs4 if isinstance(m, HumanMessage))
assert n_tool == sum(1 for m in msgs4 if isinstance(m, ToolMessage))
assert n_ai == sum(1 for m in msgs4 if isinstance(m, AIMessage))
assert n_human == 1
assert n_tool >= 1                 # 도구가 최소 한 번 불렸다
assert n_ai >= 2                   # 도구를 부르는 AI 메시지 + 최종 답 AI 메시지
print('✅ 통과!')

**서술 답안** — 아래에 각 메시지 종류의 역할을 적으세요.

*(여기에 HumanMessage·AIMessage·ToolMessage 가 각각 무엇을 담는지 서술하세요)*

**모범 서술**: **HumanMessage** 는 사용자의 질문을 담습니다. **AIMessage** 는 두 번 나오는데, 첫 번째는 모델이 **어떤 도구를 어떤 인자로 부를지 결정**(`.tool_calls`)한 메시지이고, 두 번째는 도구 결과를 반영한 **최종 자연어 답**입니다. **ToolMessage** 는 그 사이에서 **도구를 실제로 실행한 결과**를 담습니다. 즉 궤적은 '질문 → 도구 부르기로 결정 → 도구 실행 결과 → 최종 답'의 흐름을 그대로 보여 줍니다.

---
수고했어요! 도구를 **정의**하고, 도구의 **명세**(모델이 읽는 이름·설명·인자)를 확인하고, 에이전트의 **궤적**(어떤 도구가 불려 어떤 결과를 냈는지)을 읽는 기초를 익혔습니다. LV2 에서는 RAG 체인을 직접 조립하고, 데이터베이스와 외부 API 를 도구로 연결합니다.